In [1]:
import gradio as gr
import os
from dotenv import dotenv_values, load_dotenv
from langchain.chat_models import init_chat_model
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders.text import TextLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.documents.base import Document
from langsmith import Client
from langgraph.graph import StateGraph, START
from typing_extensions import List, TypedDict
from IPython.display import Image, display
import langchain_fireworks


c:\ProgramData\anaconda3\envs\moodle-assistant\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
load_dotenv()
config = dotenv_values()
FIREWORKS_API_KEY = config.get("FIREWORKS_API_KEY")
LANGSMITH_API_KEY = config.get("LANGSMITH_API_KEY")
client = Client(api_key=LANGSMITH_API_KEY)
prompt = client.pull_prompt("rlm/rag-prompt", include_model=True)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"
)
llm = init_chat_model(
    "accounts/fireworks/models/llama-v3p1-70b-instruct", model_provider="fireworks"
)
current_dir = os.getcwd()
history = gr.State([])
sem_chunker = SemanticChunker(embeddings, breakpoint_threshold_type="percentile")


In [25]:
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str


def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    message = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(message)
    return {"answer": response.content}


def load_chunk_text(text_path:str="./text_document.txt", chunker:SemanticChunker=sem_chunker) -> List[Document]:
    with open(text_path, "r") as f:
        text = f.read()
    docs = chunker.create_documents(texts=[text])
    return docs

In [26]:
all_splits = load_chunk_text()
_ = vector_store.add_documents(documents=all_splits)

In [27]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [28]:
async for message, metadata in graph.astream(
    {"question": "How can one better transmit glassblowing knowledge to novices?"}, stream_mode="messages"
):
    print(message.content, sep=' ')


One effective
 way to transmit glassblowing knowledge to novices is to have
 apprentices from previous cohorts help articulate
 and verbalize techniques, creating a bridge between tacit and explicit knowledge.
 This approach can help to clarify the
 precise steps involved in techniques such
 as gathering glass from the furnace. Additionally, acknowledging that
 individual learning preferences vary considerably can help instructors tailor
 their teaching to meet the
 needs of different learners
.



# Video Segmentation

This part of the tests are dedicated to segmenting either manually or automatically some videos for targeted elicitation purposes.